# 02 — Search Papers by Keyword via OpenAlex

This notebook demonstrates how to:
1. Search OpenAlex for papers matching a keyword or phrase
2. Filter by open access status, publication year, or type
3. Paginate through results using OpenAlex's cursor-based pagination
4. Collect results into a DataFrame

**API docs:** https://docs.openalex.org/api-entities/works/search-works

In [1]:
import requests
import pandas as pd
import time

In [2]:
MAILTO = 'your.email@example.com'  # replace with your e-mail
BASE_URL = 'https://api.openalex.org/works'

## 1. Basic keyword search

In [3]:
params = {
    'search': 'machine learning africa',
    'per_page': 10,
    'mailto': MAILTO,
}

response = requests.get(BASE_URL, params=params)
data = response.json()

print(f"Total matching works: {data['meta']['count']}")
print(f"Returned this page : {len(data['results'])}")

Total matching works: 170073
Returned this page : 10


In [4]:
# Quick look at titles
for i, work in enumerate(data['results'], 1):
    print(f"{i:2d}. [{work.get('publication_year')}] {work.get('title', 'No title')[:90]}")

 1. [2021] African soil properties and nutrients mapped at 30 m spatial resolution using two-scale en
 2. [2017] Soil nutrient maps of Sub-Saharan Africa: assessment of soil nutrient content at 250 m spa
 3. [2022] Crops yield prediction based on machine learning models: Case of West African countries
 4. [2021] Exploring teachers' preconceptions of teaching machine learning in high school: A prelimin
 5. [2020] Evaluating Machine Learning Techniques for Detecting Offensive and Hate Speech in South Af
 6. [2020] Forecasting Hourly Global Horizontal Solar Irradiance in South Africa Using Machine Learni
 7. [2022] Agricultural decision system based on advanced machine learning models for yield predictio
 8. [2021] Large-Scale High-Resolution Coastal Mangrove Forests Mapping Across West Africa With Machi
 9. [2011] High-dimensional pharmacogenetic prediction of a continuous trait using machine learning t
10. [2019] Evaluating Combinations of Sentinel-2 Data and Machine-Learning Algorithms

## 2. Add filters: open access only, year range

In [5]:
# Filters are combined with commas; ranges use colons
params_filtered = {
    'search': 'machine learning africa',
    'filter': 'open_access.is_oa:true,publication_year:2020-2023',
    'per_page': 10,
    'mailto': MAILTO,
}

response = requests.get(BASE_URL, params=params_filtered)
data_filtered = response.json()

print(f"Filtered results (OA, 2020-2023): {data_filtered['meta']['count']}")

Filtered results (OA, 2020-2023): 78016


In [6]:
for i, work in enumerate(data_filtered['results'], 1):
    oa = work.get('open_access', {}).get('oa_status', 'unknown')
    print(f"{i:2d}. [{work.get('publication_year')}][{oa:8s}] {work.get('title', '')[:80]}")

 1. [2021][gold    ] African soil properties and nutrients mapped at 30 m spatial resolution using tw
 2. [2022][gold    ] Crops yield prediction based on machine learning models: Case of West African co
 3. [2021][gold    ] Exploring teachers' preconceptions of teaching machine learning in high school: 
 4. [2020][gold    ] Evaluating Machine Learning Techniques for Detecting Offensive and Hate Speech i
 5. [2020][gold    ] Forecasting Hourly Global Horizontal Solar Irradiance in South Africa Using Mach
 6. [2022][gold    ] Agricultural decision system based on advanced machine learning models for yield
 7. [2021][gold    ] Large-Scale High-Resolution Coastal Mangrove Forests Mapping Across West Africa 
 8. [2020][bronze  ] Evaluating machine learning algorithms for predicting maize yield under conserva
 9. [2023][gold    ] Application of deep learning and machine learning models to improve healthcare i
10. [2021][gold    ] Use of machine learning techniques to identify HIV predictors

## 3. Cursor-based pagination

OpenAlex returns a `next_cursor` value in `meta` that you pass as the `cursor` parameter in the next request. Use `cursor=*` to start from the beginning.

In [7]:
def search_all_works(query: str, filters: str = '', max_results: int = 50, mailto: str = '') -> list:
    """Fetch up to max_results works matching the query, using cursor pagination."""
    results = []
    cursor = '*'

    while len(results) < max_results:
        params = {
            'search': query,
            'per_page': min(25, max_results - len(results)),
            'cursor': cursor,
            'mailto': mailto,
        }
        if filters:
            params['filter'] = filters

        response = requests.get(BASE_URL, params=params)
        data = response.json()
        page_results = data.get('results', [])

        if not page_results:
            break

        results.extend(page_results)
        cursor = data.get('meta', {}).get('next_cursor')

        if not cursor:
            break

        time.sleep(0.2)  # be polite

    print(f'Fetched {len(results)} works')
    return results

In [8]:
works = search_all_works(
    query='machine learning africa',
    filters='open_access.is_oa:true,publication_year:2020-2023',
    max_results=30,
    mailto=MAILTO,
)

Fetched 30 works


## 4. Collect results into a DataFrame

In [9]:
def works_to_dataframe(works: list) -> pd.DataFrame:
    rows = []
    for w in works:
        authorships = w.get('authorships', [])
        authors = '; '.join(
            a['author']['display_name']
            for a in authorships[:3]  # first three authors
            if a.get('author')
        )
        rows.append({
            'openalex_id': w.get('id', '').split('/')[-1],
            'doi': w.get('doi', ''),
            'title': w.get('title', ''),
            'year': w.get('publication_year'),
            'cited_by_count': w.get('cited_by_count'),
            'is_oa': w.get('open_access', {}).get('is_oa'),
            'oa_status': w.get('open_access', {}).get('oa_status'),
            'authors_preview': authors,
            'type': w.get('type'),
        })
    return pd.DataFrame(rows)


df = works_to_dataframe(works)
print(df.shape)
df.head(10)

(30, 9)


,openalex_id,doi,title,year,cited_by_count,is_oa,oa_status,authors_preview,type
0,W3111077008,https://doi.org/10.1038/s41598-021-85639-y,African soil properties and nutrients mapped a...,2021,258,True,gold,Tomislav Hengl; Matt Miller; Josip Križan,article
1,W4225556963,https://doi.org/10.1016/j.atech.2022.100049,Crops yield prediction based on machine learni...,2022,163,True,gold,Lontsi Saadio Cedric; Wilfried Yves Hamilton A...,article
2,W4200401025,https://doi.org/10.1016/j.caeo.2021.100072,Exploring teachers' preconceptions of teaching...,2021,127,True,gold,Ismaila Temitayo Sanusi; Solomon Sunday Oyeler...,article
3,W3002426859,https://doi.org/10.1109/access.2020.2968173,Evaluating Machine Learning Techniques for Det...,2020,101,True,gold,Oluwafemi Oriola; Eduan Kotzé,article
4,W3095652165,https://doi.org/10.1109/access.2020.3034690,Forecasting Hourly Global Horizontal Solar Irr...,2020,84,True,gold,Tendani Mutavhatsindi; Caston Sigauke; Rendani...,article
5,W4220984994,https://doi.org/10.1016/j.atech.2022.100048,Agricultural decision system based on advanced...,2022,86,True,gold,Rubby Aworka; Lontsi Saadio Cedric; Wilfried Y...,article
6,W3122449402,https://doi.org/10.3389/feart.2020.560933,Large-Scale High-Resolution Coastal Mangrove F...,2021,90,True,gold,Liu Xue; Temilola Fatoyinbo; Nathan Thomas,article
7,W3018215099,https://doi.org/10.1007/s42452-020-2711-6,Evaluating machine learning algorithms for pre...,2020,60,True,bronze,Walter Mupangwa; Lovemore Chipindu; Isaiah Nya...,article
8,W4386374624,https://doi.org/10.1016/j.teler.2023.100097,Application of deep learning and machine learn...,2023,56,True,gold,Elliot Mbunge; John Batani,article
9,W3187870935,https://doi.org/10.1186/s12874-021-01346-2,Use of machine learning techniques to identify...,2021,67,True,gold,Charles K. Mutai; Patrick McSharry; Innocent N...,article


In [10]:
# Summary statistics
print('OA status breakdown:')
print(df['oa_status'].value_counts())
print()
print('Publication year breakdown:')
print(df['year'].value_counts().sort_index())

OA status breakdown:
oa_status
gold       17
hybrid      6
bronze      3
green       2
diamond     2
Name: count, dtype: int64

Publication year breakdown:
year
2020     5
2021    11
2022     9
2023     5
Name: count, dtype: int64
